In [ ]:
import os
import re
import pandas as pd
from typing import List, Pattern

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8_Test\All_Config_Files"
#OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Config_Files_List_ShallowC.csv"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_Config_Files_List_ShallowC.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)


def compile_any(patterns: List[str], flags=re.I|re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]


# === DETECTION KEYWORDS ===
REAL_DEVICE_KEYWORDS = [
    'adb devices', 'adb get-state', 'adb get-serialno', 'adb install', 'adb install -r',
    'adb -s', 'adb shell', 'adb root', 'adb shell settings', 'adb shell input', 'adb shell pm grant'
]

EMULATOR_KEYWORDS = [
    'emulator',                   # Only if used to launch (not just mentioned)
    'android-wait-for-emulator', # Utility script for emulator wait
    'start-emulator.sh',         # Custom wrapper
    'avdmanager create avd',     # Explicit AVD setup
    'emulator -avd',             # Launch emulator
    'emulator @',                # Launch by name
]


THIRD_PARTY_KEYWORDS = [
    'gcloud firebase test android run', 'browserstack', 'saucectl', 'bstack', 'appcenter test run',
    'test_matrix.json', 'firebase.json'
]

INSTRUMENTATION_TRIGGER_KEYWORDS = [
    'adb shell am instrument', 'am instrument', './gradlew connectedandroidtest',
    'connectedcheck', 'connectedflavortest', 'createinstrumentationtestcoveragereport',
    'runinstrumentationtests', 'executescreenshottests', 'orchestrator', 'connectedtest'
]

UNIT_TEST_KEYWORDS_CI = [
    'gradlew test', './gradlew test', './gradlew jvmtest',
    'testdebugunittest', 'testreleaseunittest', 'kotlintest',
    'unittest', 'run unit tests', 'run: test', 'npm test', 'yarn test'
]

UNIT_TEST_KEYWORDS_BUILD = ['junit', 'testimplementation']
#INSTRUMENT_TEST_KEYWORDS_BUILD = ['androidtestimplementation', 'espresso', 'uiautomator']
INSTRUMENT_TEST_KEYWORDS_BUILD = [
    # Dependency configurations (case-insensitive)
    "androidtestimplementation",   # already have
    "androidandroidtestimplementation",  # rare typo variant, but appears in some repos
    "androidtestapi",
    "androidtestcompile",          # legacy

    # Common test libraries
    "androidx.test",               # catches all androidx.test.* deps
    "test.ext:junit",               # androidx.test.ext:junit
    "test.espresso",                # espresso-core, espresso-intents, etc.
    "test.uiautomator",             # already have uiautomator, but make it broader
    "espresso-core",
    "espresso-intents",
    "espresso-web",
    "espresso-contrib",
    "uiautomator",
    "orchestrator",                 # test orchestrator

    # Compose instrumentation test libs
    "compose.ui.test",              # catches junit4 and manifest
    "compose.ui:ui-test",
    "compose.ui:ui-test-junit4",
    "compose.ui:ui-test-manifest",

    # Google test services
    "play-services-test",
    "firebase-testlab",             # Firebase Test Lab integration

    # Test runners & rules
    "test.runner",
    "test.rules",

    # Other instrumentation frameworks
    "barista",                      # Barista UI testing lib
    "kaspresso",                    # Kaspresso UI testing lib
    "shot-android",                 # Karumi Shot screenshot tests

    # Gradle config indicators
    "testinstrumentationrunner",    # key property
    "manageddevices",               # Gradle Managed Devices
    "devicegroups",                 # GMD device groups
    "testoptions",                  # often wraps managed devices or runner args
]


STRICT_CI_PLATFORMS = ['github_actions', 'gitlab', 'jenkins', 'azure']
LENIENT_CI_PLATFORMS = ['travis_ci', 'bitrise', 'circle_ci', 'appveyor', 'teamcity', 'buddy']

# === MAIN ANALYSIS ===
results = []

for file in os.listdir(CONFIG_DIR):
    file_path = os.path.join(CONFIG_DIR, file)
    if not os.path.isfile(file_path):
        continue

    file_ext = os.path.splitext(file)[-1].lower()
    if file_ext not in ['.sh', '.json', '.gradle', '.kts', '.yml', '.yaml']:
        continue

    full_name, ci_platform = "Unknown", "Unknown"
    if "__" in file and "++" in file:
        try:
            full_name = file.split("__")[0]
            ci_platform = file.split("__")[1].split("++")[0]
        except:
            pass

    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().lower()

        parsed_ok = True
        file_is_build = file_ext in ['.gradle', '.kts']

        # === INIT VALUES ===
        device_setup = set()
        trigger_detected = False
        test_definition = False
        unit_test_ci = False
        unit_test_build = False
        instru_test_status = "None"

        # === BUILD FILES: Only Analyze Test Definition & Build Unit Test ===
        if file_is_build:
            if any(k in content for k in UNIT_TEST_KEYWORDS_BUILD):
                unit_test_build = True
            if any(k in content for k in INSTRUMENT_TEST_KEYWORDS_BUILD):
                test_definition = True

        else:
            # === DEVICE SETUP ===
            if any(k in content for k in REAL_DEVICE_KEYWORDS):
                device_setup.add("Real_Device")
            if any(k in content for k in EMULATOR_KEYWORDS):
                device_setup.add("Emulator")
            if any(k in content for k in THIRD_PARTY_KEYWORDS):
                device_setup.add("Third_Party_Lab")

            # === TEST TRIGGER ===
            if any(k in content for k in INSTRUMENTATION_TRIGGER_KEYWORDS):
                trigger_detected = True

            # === UNIT TEST (CI) ===
            if any(k in content for k in UNIT_TEST_KEYWORDS_CI):
                unit_test_ci = True

            # === STATUS LOGIC ===
            platform_key = ci_platform.strip().lower()
            ds = "None" if not device_setup else ", ".join(sorted(device_setup))
            ht = "Yes" if trigger_detected else "No"

            if file_ext in ['.yml', '.yaml']:
                if ds == "None" and ht == "Yes":
                    if platform_key in STRICT_CI_PLATFORMS:
                        instru_test_status = "Defective"
                    elif platform_key in LENIENT_CI_PLATFORMS:
                        instru_test_status = "Complete"
                elif ds != "None" and ht == "Yes":
                    instru_test_status = "Complete"
                elif ds != "None" and ht == "No":
                    instru_test_status = "Manual"


    except Exception as e:
        parsed_ok = False
        ds = "None"
        ht = "No"
        instru_test_status = "Error while parsing"

    # === FINALIZE FIELDS FOR EXPORT ===
    results.append({
        'filename': file,
        'file_type': file_ext[1:],
        'full_name': full_name,
        'ci_platform': ci_platform,
        'device_setup': "None" if file_is_build else (", ".join(sorted(device_setup)) if device_setup else "None"),
        'has_trigger': "No" if file_is_build else ("Yes" if trigger_detected else "No"),
        'has_test_definition': "Yes" if test_definition else "No",
        'unit_test_ci': False if file_is_build else unit_test_ci,
        'unit_test_build': unit_test_build,
        'instru_test_status': instru_test_status,
        'parsed_ok': parsed_ok
    })

# === EXPORT RESULTS ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Instrumentation analysis complete! Results saved to:\n{OUTPUT_CSV}")


✅ Instrumentation analysis complete! Results saved to:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_Config_Files_List_ShallowC.csv


In [4]:
import os
import re
import pandas as pd

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Config_Files_List_ShallowC.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# === DETECTION KEYWORDS ===
REAL_DEVICE_KEYWORDS = [
    'adb devices', 'adb get-state', 'adb get-serialno', 'adb install', 'adb install -r',
    'adb -s', 'adb shell', 'adb root', 'adb shell settings', 'adb shell input', 'adb shell pm grant'
]

EMULATOR_KEYWORDS = [
    'emulator',
    'android-wait-for-emulator',
    'start-emulator.sh',
    'avdmanager create avd',
    'emulator -avd',
    'emulator @',
]

THIRD_PARTY_KEYWORDS = [
    'gcloud firebase test android run', 'browserstack', 'saucectl', 'bstack', 'appcenter test run',
    'test_matrix.json', 'firebase.json'
]

INSTRUMENTATION_TRIGGER_KEYWORDS = [
    'adb shell am instrument', 'am instrument', './gradlew connectedandroidtest',
    'connectedcheck', 'connectedflavortest', 'createinstrumentationtestcoveragereport',
    'runinstrumentationtests', 'executescreenshottests', 'orchestrator', 'connectedtest'
]

UNIT_TEST_KEYWORDS_CI = [
    'gradlew test', './gradlew test', './gradlew jvmtest',
    'testdebugunittest', 'testreleaseunittest', 'kotlintest',
    'unittest', 'run unit tests', 'run: test', 'npm test', 'yarn test'
]

UNIT_TEST_KEYWORDS_BUILD = ['junit', 'testimplementation']
INSTRUMENT_TEST_KEYWORDS_BUILD = ['androidtestimplementation', 'espresso', 'uiautomator']

STRICT_CI_PLATFORMS = ['github_actions', 'gitlab', 'jenkins', 'azure']
LENIENT_CI_PLATFORMS = ['travis_ci', 'bitrise', 'circle_ci', 'appveyor', 'teamcity', 'buddy']

# === HELPERS ===
def unique_hits(keywords, content_lc):
    """Return ordered unique keyword matches (literal substring)."""
    seen = set()
    hits = []
    for k in keywords:
        k_lc = k.lower()
        if k_lc in content_lc and k not in seen:
            seen.add(k)
            hits.append(k)
    return hits

# === MAIN ANALYSIS ===
results = []

for file in os.listdir(CONFIG_DIR):
    file_path = os.path.join(CONFIG_DIR, file)
    if not os.path.isfile(file_path):
        continue

    file_ext = os.path.splitext(file)[-1].lower()
    if file_ext not in ['.sh', '.json', '.gradle', '.kts', '.yml', '.yaml']:
        continue

    full_name, ci_platform = "Unknown", "Unknown"
    if "__" in file and "++" in file:
        try:
            full_name = file.split("__")[0]
            ci_platform = file.split("__")[1].split("++")[0]
        except Exception:
            pass

    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        content_lc = content.lower()

        parsed_ok = True
        file_is_build = file_ext in ['.gradle', '.kts']

        # === INIT VALUES ===
        device_setup = set()
        trigger_detected = False
        trigger_list = []
        test_definition_list = []
        test_definition = False
        unit_test_ci = False
        unit_test_build = False
        instru_test_status = "None"

        # === BUILD FILES: detect test definitions & build unit tests ===
        if file_is_build:
            if any(k in content_lc for k in UNIT_TEST_KEYWORDS_BUILD):
                unit_test_build = True
            # collect instrumentation test definitions
            test_definition_list = unique_hits(INSTRUMENT_TEST_KEYWORDS_BUILD, content_lc)
            if test_definition_list:
                test_definition = True

        else:
            # === DEVICE SETUP ===
            if any(k in content_lc for k in REAL_DEVICE_KEYWORDS):
                device_setup.add("Real_Device")
            if any(k in content_lc for k in EMULATOR_KEYWORDS):
                device_setup.add("Emulator")
            if any(k in content_lc for k in THIRD_PARTY_KEYWORDS):
                device_setup.add("Third_Party_Lab")

            # === TEST TRIGGER(S) ===
            trigger_list = unique_hits(INSTRUMENTATION_TRIGGER_KEYWORDS, content_lc)
            trigger_detected = len(trigger_list) > 0

            # === UNIT TEST (CI) ===
            if any(k in content_lc for k in UNIT_TEST_KEYWORDS_CI):
                unit_test_ci = True

            # === STATUS LOGIC ===
            platform_key = ci_platform.strip().lower()
            ds = "None" if not device_setup else ", ".join(sorted(device_setup))
            ht = "Yes" if trigger_detected else "No"

            if file_ext in ['.yml', '.yaml']:
                if ds == "None" and ht == "Yes":
                    if platform_key in STRICT_CI_PLATFORMS:
                        instru_test_status = "Defective"
                    elif platform_key in LENIENT_CI_PLATFORMS:
                        instru_test_status = "Complete"
                elif ds != "None" and ht == "Yes":
                    instru_test_status = "Complete"
                elif ds != "None" and ht == "No":
                    instru_test_status = "Manual"

    except Exception as e:
        parsed_ok = False
        device_setup = set()
        trigger_detected = False
        trigger_list = []
        test_definition_list = []
        test_definition = False
        unit_test_ci = False
        unit_test_build = False
        instru_test_status = "Error while parsing"

    # --- finalize fields derived from sets/lists ---
    device_setup_str = "None" if not device_setup or file_is_build else ", ".join(sorted(device_setup))
    has_device_setup = "Yes" if device_setup_str != "None" else "No"

    test_trigger_str = "none" if file_is_build or not trigger_list else ", ".join(trigger_list)
    test_definition_str = "none" if not test_definition_list else ", ".join(test_definition_list)

    # === FINALIZE FIELDS FOR EXPORT ===
    results.append({
        'filename': file,
        'file_type': file_ext[1:],
        'full_name': full_name,
        'ci_platform': ci_platform,
        'has_test_definition': "Yes" if test_definition else "No",
        'has_device_setup': has_device_setup,          # (1) NEW
        'has_trigger': "No" if file_is_build else ("Yes" if trigger_detected else "No"),
        'test_definition': test_definition_str,         # (3) NEW
        'device_setup': device_setup_str,
        'test_trigger': test_trigger_str,               # (2) NEW
        'unit_test_ci': False if file_is_build else unit_test_ci,
        'unit_test_build': unit_test_build,
        'instru_test_status': instru_test_status

    })

# === EXPORT RESULTS ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Instrumentation analysis complete! Results saved to:\n{OUTPUT_CSV}")


✅ Instrumentation analysis complete! Results saved to:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Config_Files_List_ShallowC.csv
